# Data dictionary: pandas types vs UCI types

Builds a data dictionary for all 56 columns of the raw PhiUSIIL CSV. For each column it shows the type pandas reads, the type the UCI page lists, and the minimum and maximum of numeric columns. Columns where the two types disagree are marked.

The table is saved to `ml/reports/data_dictionary.md`.

Get the CSV first by following [`ml/README.md`](../README.md). Links in the data are only ever handled as text.

In [1]:
from pathlib import Path

import pandas as pd

CSV_NAME = "PhiUSIIL_Phishing_URL_Dataset.csv"


def find_ml_dir(start: Path) -> Path:
    """Walk up to the repo root (the folder holding AGENTS.md), so the notebook runs from any working directory inside the repo."""
    for folder in [start, *start.parents]:
        if (folder / "AGENTS.md").exists() and (folder / "ml").is_dir():
            return folder / "ml"
    raise FileNotFoundError("Could not find the ml/ folder. Open this notebook from inside the LinkGuard repo.")


ML_DIR = find_ml_dir(Path.cwd().resolve())
DATA_PATH = ML_DIR / "data" / CSV_NAME
REPORT_PATH = ML_DIR / "reports" / "data_dictionary.md"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download it by following 'Get the dataset' in ml/README.md, then run this notebook again."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")

Loaded 235,795 rows and 56 columns


## UCI types

Copied from the UCI variables table for dataset 967 (PhiUSIIL Phishing URL), so this notebook runs offline. Column names are kept exactly as UCI and the file spell them, typos included (`Degit`, `Spacial`).

In [2]:
UCI_TYPES = {
    "FILENAME": "Categorical",
    "URL": "Categorical",
    "URLLength": "Integer",
    "Domain": "Categorical",
    "DomainLength": "Integer",
    "IsDomainIP": "Integer",
    "TLD": "Categorical",
    "URLSimilarityIndex": "Integer",
    "CharContinuationRate": "Integer",
    "TLDLegitimateProb": "Continuous",
    "URLCharProb": "Continuous",
    "TLDLength": "Integer",
    "NoOfSubDomain": "Integer",
    "HasObfuscation": "Integer",
    "NoOfObfuscatedChar": "Integer",
    "ObfuscationRatio": "Integer",
    "NoOfLettersInURL": "Integer",
    "LetterRatioInURL": "Continuous",
    "NoOfDegitsInURL": "Integer",
    "DegitRatioInURL": "Integer",
    "NoOfEqualsInURL": "Integer",
    "NoOfQMarkInURL": "Integer",
    "NoOfAmpersandInURL": "Integer",
    "NoOfOtherSpecialCharsInURL": "Integer",
    "SpacialCharRatioInURL": "Continuous",
    "IsHTTPS": "Integer",
    "LineOfCode": "Integer",
    "LargestLineLength": "Integer",
    "HasTitle": "Integer",
    "Title": "Categorical",
    "DomainTitleMatchScore": "Integer",
    "URLTitleMatchScore": "Integer",
    "HasFavicon": "Integer",
    "Robots": "Integer",
    "IsResponsive": "Integer",
    "NoOfURLRedirect": "Integer",
    "NoOfSelfRedirect": "Integer",
    "HasDescription": "Integer",
    "NoOfPopup": "Integer",
    "NoOfiFrame": "Integer",
    "HasExternalFormSubmit": "Integer",
    "HasSocialNet": "Integer",
    "HasSubmitButton": "Integer",
    "HasHiddenFields": "Integer",
    "HasPasswordField": "Integer",
    "Bank": "Integer",
    "Pay": "Integer",
    "Crypto": "Integer",
    "HasCopyrightInfo": "Integer",
    "NoOfImage": "Integer",
    "NoOfCSS": "Integer",
    "NoOfJS": "Integer",
    "NoOfSelfRef": "Integer",
    "NoOfEmptyRef": "Integer",
    "NoOfExternalRef": "Integer",
    "label": "Integer",
}

print(f"{len(UCI_TYPES)} UCI columns")

56 UCI columns


In [3]:
missing_from_file = [name for name in UCI_TYPES if name not in df.columns]
not_in_uci = [name for name in df.columns if name not in UCI_TYPES]
if missing_from_file or not_in_uci:
    raise ValueError(f"Column names differ from UCI. Missing from file: {missing_from_file}. Not in UCI: {not_in_uci}.")

print(f"All {len(UCI_TYPES)} column names match UCI exactly.")

All 56 column names match UCI exactly.


## Build the dictionary

pandas types map to UCI types like this:

| pandas reads | Counts as UCI type |
|---|---|
| `int64` | Integer |
| `float64` | Continuous |
| text (`str` / `object`) | Categorical |

A column is marked when the UCI type pandas implies is not the one UCI lists. Minimum and maximum are only given for numeric columns.

In [4]:
def implied_uci_type(series: pd.Series) -> str:
    if pd.api.types.is_integer_dtype(series):
        return "Integer"
    if pd.api.types.is_float_dtype(series):
        return "Continuous"
    return "Categorical"


def format_number(value) -> str:
    return f"{value:g}" if isinstance(value, float) else str(value)


def dictionary_row(name: str) -> dict:
    series = df[name]
    is_numeric = pd.api.types.is_numeric_dtype(series)
    uci_type = UCI_TYPES[name]
    return {
        "Column": name,
        "pandas type": str(series.dtype),
        "UCI type": uci_type,
        "Types differ": "YES" if implied_uci_type(series) != uci_type else "",
        "Min": format_number(series.min()) if is_numeric else "",
        "Max": format_number(series.max()) if is_numeric else "",
    }


dictionary = pd.DataFrame([dictionary_row(name) for name in df.columns])
mismatches = dictionary[dictionary["Types differ"] == "YES"]

print(f"{len(dictionary)} columns, {len(mismatches)} where pandas and UCI types differ:\n")
print(mismatches[["Column", "pandas type", "UCI type", "Min", "Max"]].to_string(index=False))

56 columns, 6 where pandas and UCI types differ:

               Column pandas type UCI type      Min   Max
   URLSimilarityIndex     float64  Integer 0.155574   100
 CharContinuationRate     float64  Integer        0     1
     ObfuscationRatio     float64  Integer        0 0.348
      DegitRatioInURL     float64  Integer        0 0.684
DomainTitleMatchScore     float64  Integer        0   100
   URLTitleMatchScore     float64  Integer        0   100


## What the mismatches mean

All 6 mismatches are columns UCI lists as **Integer** that actually hold decimals, so pandas reads them as `float64`:

- **Ratios, between 0 and 1:** `ObfuscationRatio`, `DegitRatioInURL`
- **A rate, between 0 and 1:** `CharContinuationRate`
- **Scores out of 100:** `URLSimilarityIndex`, `DomainTitleMatchScore`, `URLTitleMatchScore`

UCI's type is wrong for these columns, and the file is right. Treat them as continuous numbers when building features and training. Casting them to integers would turn every ratio and rate below 1 into 0.

## Full dictionary

In [5]:
print(dictionary.to_string(index=False))

                    Column pandas type    UCI type Types differ        Min       Max
                  FILENAME         str Categorical                                  
                       URL         str Categorical                                  
                 URLLength       int64     Integer                      13      6097
                    Domain         str Categorical                                  
              DomainLength       int64     Integer                       4       110
                IsDomainIP       int64     Integer                       0         1
                       TLD         str Categorical                                  
        URLSimilarityIndex     float64     Integer          YES   0.155574       100
      CharContinuationRate     float64     Integer          YES          0         1
         TLDLegitimateProb     float64  Continuous                       0  0.522907
               URLCharProb     float64  Continuous              0

## Save it

In [6]:
def to_markdown_table(table: pd.DataFrame) -> str:
    header = "| " + " | ".join(table.columns) + " |"
    divider = "|" + "|".join("---" for _ in table.columns) + "|"
    rows = [
        f"| `{row[0]}` | " + " | ".join(str(cell) for cell in row[1:]) + " |"
        for row in table.itertuples(index=False)
    ]
    return "\n".join([header, divider, *rows])


report = f"""# PhiUSIIL data dictionary

Generated by `ml/notebooks/02_data_dictionary.ipynb` from the raw file `{CSV_NAME}` ({len(df):,} rows, {df.shape[1]} columns).
UCI types come from the UCI variables table for dataset 967.

**{len(mismatches)} of {len(dictionary)} columns** have a pandas type that differs from the UCI type. They are marked YES below.

{to_markdown_table(dictionary)}
"""

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Saved {REPORT_PATH.relative_to(ML_DIR.parent)}")

Saved ml/reports/data_dictionary.md
